# 🎯 NRA Target Scorer — Cloud GPU Training Pipeline (Phase 4)

Welcome to the Google Colab training environment for the **Camera-Independent NRA Pistol Target Scorer**!

### What This Notebook Does:
1. Verifies cloud GPU acceleration (NVIDIA T4 / A100 / L4) and Tensor Cores.
2. Automatically unpacks and installs the `target_scorer` package.
3. Runs procedural training with mixed-precision (AMP) acceleration.
4. Generates an interactive visual inspection scorecard right in your browser.
5. Directly downloads the optimal checkpoint (`target_scoring_hrnet_best.pt`) to your local machine.

---
### Step 1: Connect to GPU Runtime
In the top menu, go to **Runtime** > **Change runtime type**, select **T4 GPU** (or A100), and click **Save**.

In [ ]:
# Check GPU hardware and VRAM
!nvidia-smi
import torch
print("\n" + "=" * 50)
print("PyTorch Version:", torch.__version__)
print("CUDA Available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:     ", torch.cuda.get_device_name(0))
    print("VRAM Capacity:  ", f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print("=" * 50)

## Step 2: Set Up Codebase

**Option A (Fastest): Upload `target_scorer_colab_bundle.zip`**
- In the left sidebar, click the **Folder icon** 📁 to open files.
- Drag and drop your local `target_scorer_colab_bundle.zip` into the file list.
- Run the cell below to unpack and install the package.

*(Or if you pushed your repository to GitHub, you can uncomment `!git clone` below)*.

In [ ]:
import os
import shutil

# If bundle was uploaded:
if os.path.exists("/content/target_scorer_colab_bundle.zip"):
    print("[*] Unpacking target_scorer_colab_bundle.zip...")
    !unzip -q -o /content/target_scorer_colab_bundle.zip -d /content/
elif os.path.exists("/content/target_scorer"):
    print("[*] Found existing target_scorer directory.")
else:
    # Optional: Clone from git repository if set
    # !git clone https://github.com/<your-user>/target_scorer.git /content/target_scorer
    print("[!] Please drag and drop target_scorer_colab_bundle.zip into the Colab Files panel on the left!")

# Navigate into the project and install in editable mode
%cd /content/target_scorer
!pip install -q -e .
print("\n[✓] Environment configured successfully!")

## Step 3: Configure Training Parameters

Select your training resolution preset:
- **Preset 1 (Ultra-Res Full Target 024 	imes 1024$ @ .0	ext{ px/mm}$):** Maximum resolution, pristine sub-pixel accuracy on A100 GPU.
- **Preset 2 (Full Target 00 	imes 800$ @ .0	ext{ px/mm}$):** Encloses all rings out to Ring 4 (86.2	ext{ mm}$ diameter) with zero edge clipping (~7 minutes on T4).
- **Preset 3 (Canonical 12 	imes 512$ @ .0	ext{ px/mm}$):** Aiming black bullseye + 6-ring (~3 minutes for 15 epochs on T4).

In [ ]:
# @title Training Hyperparameters { run: "auto" }
PRESET = "Full Target B-2 (800x800 @ 4.0 px/mm)" # @param ["Ultra-Res Full Target (1024x1024 @ 4.0 px/mm)", "Full Target B-2 (800x800 @ 4.0 px/mm)", "Canonical Bullseye (512x512 @ 4.0 px/mm)", "Low-Res Quick Test (256x256 @ 2.0 px/mm)"]
EPOCHS = 15 # @param {type:"integer"}
BATCH_SIZE = 8 # @param {type:"integer"}
TRAIN_SAMPLES_PER_EPOCH = 400 # @param {type:"integer"}
VAL_SAMPLES_PER_EPOCH = 50 # @param {type:"integer"}
LEARNING_RATE = 3.5e-4 # @param {type:"number"}
USE_AMP = True # @param {type:"boolean"}

if "1024x1024" in PRESET:
    CANVAS_SIZE = 1024
    SCALE = 4.0
elif "800x800" in PRESET:
    CANVAS_SIZE = 800
    SCALE = 4.0
elif "512x512" in PRESET:
    CANVAS_SIZE = 512
    SCALE = 4.0
else:
    CANVAS_SIZE = 256
    SCALE = 2.0

CHECKPOINT_PATH = f"/content/target_scorer/weights/target_scoring_hrnet_{CANVAS_SIZE}_best.pt"

print(f"[*] Configured: {CANVAS_SIZE}x{CANVAS_SIZE} @ {SCALE} px/mm | Batch: {BATCH_SIZE} | Epochs: {EPOCHS} | AMP: {USE_AMP}")
print(f"[*] Checkpoint destination: {CHECKPOINT_PATH}")

## Step 4: Run Training on Cloud GPU

Click run below to train the U-Net heatmap model with Adaptive Wing Loss on the cloud GPU:

In [ ]:
# Ensure OpenCV float16 compatibility under AMP
import os, numpy as np
patch_file = "/content/target_scorer/src/target_scorer/metrology/peaks.py"
if os.path.exists(patch_file):
    with open(patch_file, "r") as f:
        _content = f.read()
    if "heatmap.dtype" not in _content:
        _content = _content.replace(
            "    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))",
            "    if heatmap.dtype != np.float32:\n        heatmap = heatmap.astype(np.float32)\n    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))"
        )
        with open(patch_file, "w") as f:
            f.write(_content)
        print("[✓] Verified and applied OpenCV AMP float16 patch to peaks.py\n")

amp_flag = "--amp" if USE_AMP else "--no-amp"

cmd = (
    f"target-scorer train "
    f"--canvas-size {CANVAS_SIZE} "
    f"--scale {SCALE} "
    f"--epochs {EPOCHS} "
    f"--batch-size {BATCH_SIZE} "
    f"--train-samples {TRAIN_SAMPLES_PER_EPOCH} "
    f"--val-samples {VAL_SAMPLES_PER_EPOCH} "
    f"--lr {LEARNING_RATE} "
    f"--checkpoint {CHECKPOINT_PATH} "
    f"{amp_flag}"
)

print(f"Executing: {cmd}\n")
!{cmd}


## Step 5: In-Browser Visual Inspection Scorecard

Run inference with the newly trained cloud weights and display the official NRA visual scorecard directly in your Colab notebook:

In [ ]:
from IPython.display import Image, display

output_png = f"/content/target_scorer/colab_inspection_{CANVAS_SIZE}.png"

!target-scorer infer \
    --checkpoint {CHECKPOINT_PATH} \
    --target B-2 \
    --canvas-size {CANVAS_SIZE} \
    --scale {SCALE} \
    --output {output_png}

print(f"\n[✓] Rendered Visual Scorecard HUD ({CANVAS_SIZE}x{CANVAS_SIZE}):")
display(Image(output_png))

## Step 6: Download Optimal Checkpoint to Your Computer

Click below to download the trained checkpoint directly to your local computer. Then copy it into your local `weights/` directory:

In [ ]:
from google.colab import files
import os

if os.path.exists(CHECKPOINT_PATH):
    print(f"[*] Downloading \x27{CHECKPOINT_PATH}\x27 to your computer...")
    files.download(CHECKPOINT_PATH)
else:
    print(f"[!] Checkpoint \x27{CHECKPOINT_PATH}\x27 not found. Did training complete?")